In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'monospace'

print("All libraries loaded ✓")

All libraries loaded ✓


In [2]:
orders = pd.read_csv('../data/olist_orders_dataset.csv')
items = pd.read_csv('../data/olist_order_items_dataset.csv')
payments = pd.read_csv('../data/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv')
customers = pd.read_csv('../data/olist_customers_dataset.csv')
sellers = pd.read_csv('../data/olist_sellers_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
geo = pd.read_csv('../data/olist_geolocation_dataset.csv')
translation = pd.read_csv('../data/product_category_name_translation.csv')

datasets = {
    'orders': orders, 'items': items, 'payments': payments,
    'reviews': reviews, 'customers': customers, 'sellers': sellers,
    'products': products, 'translation': translation
}

for name, df in datasets.items():
    print(f"{name:15} → {df.shape[0]:,} rows × {df.shape[1]} cols")

orders          → 99,441 rows × 8 cols
items           → 112,650 rows × 7 cols
payments        → 103,886 rows × 5 cols
reviews         → 99,224 rows × 7 cols
customers       → 99,441 rows × 5 cols
sellers         → 3,095 rows × 4 cols
products        → 32,951 rows × 9 cols
translation     → 71 rows × 2 cols


In [3]:
date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'])
print("Datetime columns parsed ✓")


Datetime columns parsed ✓


In [4]:
print(f"Total orders: {len(orders):,}")
print(orders['order_status'].value_counts())
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()
print(f"\nDelivered orders only: {len(orders_delivered):,}")
print(f"Dropped: {len(orders) - len(orders_delivered):,} rows ({((len(orders) - len(orders_delivered))/len(orders)*100):.1f}%)")

Total orders: 99,441
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Delivered orders only: 96,478
Dropped: 2,963 rows (3.0%)


In [5]:
df = orders_delivered.merge(items, on='order_id', how='left')
df = df.merge(payments.groupby('order_id').agg(
    payment_value=('payment_value', 'sum'),
    payment_installments=('payment_installments', 'max'),
    payment_type=('payment_type', 'first')
).reset_index(), on='order_id', how='left')
df = df.merge(reviews[['order_id','review_score']].drop_duplicates('order_id'), on='order_id', how='left')
df = df.merge(customers, on='customer_id', how='left')
df = df.merge(products, on='product_id', how='left')
df = df.merge(translation, on='product_category_name', how='left')
df = df.merge(sellers[['seller_id','seller_state','seller_city']], on='seller_id', how='left')
print(f"Master dataframe: {df.shape}")
print(df.columns.tolist())

Master dataframe: (110197, 33)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'payment_value', 'payment_installments', 'payment_type', 'review_score', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'seller_state', 'seller_city']


In [6]:
df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df['on_time'] = df['order_delivered_customer_date'] <= df['order_estimated_delivery_date']
df['order_year'] = df['order_purchase_timestamp'].dt.year
df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)
df['order_month_num'] = df['order_purchase_timestamp'].dt.month
df['order_dayofweek'] = df['order_purchase_timestamp'].dt.day_name()
df['order_hour'] = df['order_purchase_timestamp'].dt.hour
df['item_revenue'] = df['price'] + df['freight_value']
print(df[['delivery_days','on_time','order_year','item_revenue']].describe())

       delivery_days  order_year  item_revenue
count      110189.00   110197.00     110197.00
mean           12.01     2017.54        139.93
std             9.45        0.50        189.32
min             0.00     2016.00          6.08
25%             6.00     2017.00         55.18
50%            10.00     2018.00         92.13
75%            15.00     2018.00        157.51
max           209.00     2018.00       6929.31


In [7]:
null_report = df.isnull().sum()
null_report = null_report[null_report > 0].sort_values(ascending=False)
null_pct = (null_report / len(df) * 100).round(2)
null_df = pd.DataFrame({'missing_count': null_report, 'missing_pct': null_pct})
print(null_df)

                               missing_count  missing_pct
product_category_name_english           1559         1.41
product_photos_qty                      1537         1.39
product_category_name                   1537         1.39
product_description_lenght              1537         1.39
product_name_lenght                     1537         1.39
review_score                             827         0.75
product_width_cm                          18         0.02
product_weight_g                          18         0.02
product_length_cm                         18         0.02
product_height_cm                         18         0.02
order_approved_at                         15         0.01
order_delivered_customer_date              8         0.01
delivery_days                              8         0.01
payment_installments                       3         0.00
payment_type                               3         0.00
payment_value                              3         0.00
order_delivere

In [8]:
fig = px.bar(
    null_df.reset_index(), x='index', y='missing_pct',
    title='Missing Data by Column (%)',
    labels={'index': 'Column', 'missing_pct': 'Missing (%)'},
    color='missing_pct', color_continuous_scale='Reds'
)
fig.update_layout(template='plotly_white', showlegend=False)
fig.show()
fig.write_html('../outputs/01_missing_data.html')

In [9]:
df['review_score'] = df['review_score'].fillna(df['review_score'].median())
df = df.dropna(subset=['delivery_days'])
df['product_category_name_english'] = df['product_category_name_english'].fillna('unknown')

In [10]:
df.to_csv('../data/olist_master_clean.csv', index=False)
print("Clean dataset saved ✓")

Clean dataset saved ✓


In [11]:
monthly_revenue = df.groupby('order_month').agg(
    revenue=('item_revenue', 'sum'),
    orders=('order_id', 'nunique')
).reset_index().sort_values('order_month')
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=monthly_revenue['order_month'], y=monthly_revenue['revenue'],
    name='Revenue (BRL)', marker_color='#1a1a2e', opacity=0.85), secondary_y=False)
fig.add_trace(go.Scatter(x=monthly_revenue['order_month'], y=monthly_revenue['orders'],
    name='Order Count', line=dict(color='#e94560', width=2.5), mode='lines+markers'), secondary_y=True)
fig.update_layout(
    title='📈 Monthly Revenue & Order Volume (2016–2018)',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.update_yaxes(title_text='Revenue (BRL)', secondary_y=False)
fig.update_yaxes(title_text='Order Count', secondary_y=True)
fig.show()
fig.write_html('../outputs/02_monthly_revenue.html')

In [12]:
bf_revenue = monthly_revenue[monthly_revenue['order_month'] == '2017-11']['revenue'].values[0]
print(f"November 2017 (Black Friday) Revenue: BRL {bf_revenue:,.0f}")
print(f"vs October 2017: BRL {monthly_revenue[monthly_revenue['order_month'] == '2017-10']['revenue'].values[0]:,.0f}")
print(f"Month-over-month spike: {((bf_revenue / monthly_revenue[monthly_revenue['order_month'] == '2017-10']['revenue'].values[0]) - 1) * 100:.1f}%")

November 2017 (Black Friday) Revenue: BRL 1,153,229
vs October 2017: BRL 751,117
Month-over-month spike: 53.5%


In [13]:
cat_revenue = df.groupby('product_category_name_english').agg(
    revenue=('item_revenue', 'sum'),
    orders=('order_id', 'nunique'),
    avg_price=('price', 'mean')
).reset_index().sort_values('revenue', ascending=False).head(10)

fig = px.bar(cat_revenue, x='revenue', y='product_category_name_english',
    orientation='h',
    color='avg_price',
    color_continuous_scale='Blues',
    title='Top 10 Product Categories by Revenue',
    labels={
        'revenue': 'Total Revenue (BRL)',
        'product_category_name_english': 'Category',
        'avg_price': 'Avg Price (BRL)'
    },
    text=cat_revenue['revenue'].apply(lambda x: f'BRL {x/1e6:.1f}M')
)
fig.update_traces(textposition='outside')
fig.update_layout(template='plotly_white', yaxis={'categoryorder': 'total ascending'})
fig.show()
fig.write_html('../outputs/03_top_categories_revenue.html')

In [14]:
# 1. Standard calculation
cat_volume = df.groupby('product_category_name_english').agg(
    revenue=('item_revenue', 'sum'),
    orders=('order_id', 'nunique')
).reset_index()

# 2. Assign ranks (1 is the best/highest)
cat_volume['revenue_rank'] = cat_volume['revenue'].rank(ascending=False)
cat_volume['volume_rank'] = cat_volume['orders'].rank(ascending=False)

# 3. FIXED FORMULA: Revenue Rank minus Volume Rank
# If Revenue Rank is 40 (poor) and Volume Rank is 2 (amazing), 40 - 2 = +38 (High margin gap!)
cat_volume['rank_gap'] = cat_volume['revenue_rank'] - cat_volume['volume_rank']

# 4. Use nlargest to get the highest positive gaps
hidden_low = cat_volume.nlargest(5, 'rank_gap')[['product_category_name_english','revenue_rank','volume_rank','rank_gap']]

print("Categories with HIGH volume but LOW revenue (low-margin cash cows):")
print(hidden_low.to_string(index=False))


Categories with HIGH volume but LOW revenue (low-margin cash cows):
product_category_name_english  revenue_rank  volume_rank  rank_gap
              books_technical         50.00        35.00     15.00
                       drinks         48.00        33.00     15.00
                         food         42.00        29.00     13.00
       books_general_interest         37.00        27.00     10.00
                   food_drink         51.00        42.00      9.00


In [15]:
aov = df.groupby('product_category_name_english')['item_revenue'].mean().reset_index()
aov.columns = ['category', 'avg_order_value']
aov = aov.sort_values('avg_order_value', ascending=False).head(15)

fig = px.bar(aov, x='avg_order_value', y='category', orientation='h',
    title='Average Order Value by Category (Top 15)',
    labels={'avg_order_value': 'Avg Order Value (BRL)', 'category': ''},
    color='avg_order_value', color_continuous_scale='Greens',
    text=aov['avg_order_value'].apply(lambda x: f'BRL {x:.0f}')
)
fig.update_traces(textposition='outside')
fig.update_layout(template='plotly_white', yaxis={'categoryorder': 'total ascending'}, showlegend=False)
fig.show()
fig.write_html('../outputs/04_avg_order_value.html')

In [16]:
fig = px.histogram(
    df[df['delivery_days'].between(0, 60)],
    x='delivery_days', nbins=60,
    title='Delivery Time Distribution (Days)',
    labels={'delivery_days': 'Days to Deliver', 'count': 'Number of Orders'},
    color_discrete_sequence=['#1a1a2e']
)
fig.add_vline(x=df['delivery_days'].median(), line_dash='dash', line_color='red',
    annotation_text=f"Median: {df['delivery_days'].median():.0f} days", annotation_position='top right')
fig.update_layout(template='plotly_white')
fig.show()
fig.write_html('../outputs/05_delivery_distribution.html')

In [17]:
on_time_rate = df['on_time'].mean() * 100
print(f"Overall On-Time Delivery Rate: {on_time_rate:.1f}%")
print(f"Late deliveries: {(~df['on_time']).sum():,} orders")

# By state
state_ontime = df.groupby('customer_state').agg(
    on_time_rate=('on_time', 'mean'),
    total_orders=('order_id', 'nunique')
).reset_index()
state_ontime['on_time_pct'] = state_ontime['on_time_rate'] * 100
state_ontime = state_ontime.sort_values('on_time_pct')

fig = px.bar(state_ontime, x='customer_state', y='on_time_pct',
    title='On-Time Delivery Rate by Customer State (%)',
    color='on_time_pct',
    color_continuous_scale='Viridis',
    labels={'customer_state': 'State', 'on_time_pct': 'On-Time Rate (%)'}
)
fig.add_hline(y=on_time_rate, line_dash='dash', line_color='black',
    annotation_text=f'National avg: {on_time_rate:.1f}%')
fig.update_layout(template='plotly_white')
fig.show()
fig.write_html('../outputs/06_ontime_by_state.html')

Overall On-Time Delivery Rate: 92.1%
Late deliveries: 8,714 orders


In [18]:
df['delivery_category'] = pd.cut(df['delivery_days'],
    bins=[0, 7, 14, 21, 30, 100],
    labels=['1 week', '2 weeks', '3 weeks', '4 weeks', '4+ weeks']
)

delivery_score = df.groupby('delivery_category', observed=True)['review_score'].mean().reset_index()

fig = px.bar(delivery_score, x='delivery_category', y='review_score',
    title='⚠️ How Delivery Speed Impacts Customer Satisfaction',
    labels={'delivery_category': 'Delivery Time', 'review_score': 'Avg Review Score (1–5)'},
    color='review_score',
    color_continuous_scale='RdYlGn',
    text=delivery_score['review_score'].apply(lambda x: f'{x:.2f}')
)
fig.update_traces(textposition='outside')
fig.update_layout(template='plotly_white', yaxis_range=[0, 5.5])
fig.show()
fig.write_html('../outputs/07_delivery_vs_satisfaction.html')

In [19]:
score_dist = df['review_score'].value_counts().sort_index().reset_index()
score_dist.columns = ['score', 'count']
score_dist['pct'] = score_dist['count'] / score_dist['count'].sum() * 100

fig = px.bar(score_dist, x='score', y='count',
    title='Customer Review Score Distribution',
    labels={'score': 'Review Score', 'count': 'Number of Reviews'},
    color='score',
    color_continuous_scale='RdYlGn',
    text=score_dist['pct'].apply(lambda x: f'{x:.1f}%')
)
fig.update_traces(textposition='outside')
fig.update_layout(template='plotly_white', showlegend=False)
fig.show()
fig.write_html('../outputs/08_review_distribution.html')


In [20]:
seller_perf = df.groupby('seller_id').agg(
    total_revenue=('item_revenue', 'sum'),
    avg_rating=('review_score', 'mean'),
    total_orders=('order_id', 'nunique'),
    avg_delivery_days=('delivery_days', 'mean')
).reset_index()

seller_perf = seller_perf[seller_perf['total_orders'] >= 10]  # min 10 orders

# Quadrant labels
med_revenue = seller_perf['total_revenue'].median()
med_rating = seller_perf['avg_rating'].median()

def label_seller(row):
    if row['total_revenue'] >= med_revenue and row['avg_rating'] >= med_rating:
        return '⭐ Star Sellers'
    elif row['total_revenue'] >= med_revenue and row['avg_rating'] < med_rating:
        return '⚠️ High Revenue, Low Satisfaction'
    elif row['total_revenue'] < med_revenue and row['avg_rating'] >= med_rating:
        return '💎 Hidden Gems'
    else:
        return '🔴 At Risk'

seller_perf['segment'] = seller_perf.apply(label_seller, axis=1)
print(seller_perf['segment'].value_counts())

fig = px.scatter(seller_perf, x='total_revenue', y='avg_rating',
    color='segment',
    size='total_orders',
    hover_data=['seller_id', 'avg_delivery_days'],
    title='Seller Performance Quadrant: Revenue vs Customer Satisfaction',
    labels={'total_revenue': 'Total Revenue (BRL)', 'avg_rating': 'Avg Review Score'},
    color_discrete_map={
        '⭐ Star Sellers': '#2ecc71',
        '⚠️ High Revenue, Low Satisfaction': '#e74c3c',
        '💎 Hidden Gems': '#3498db',
        '🔴 At Risk': '#95a5a6'
    }
)
fig.add_vline(x=med_revenue, line_dash='dot', line_color='gray')
fig.add_hline(y=med_rating, line_dash='dot', line_color='gray')
fig.update_layout(template='plotly_white')
fig.show(renderer="browser")
fig.write_html('../outputs/09_seller_quadrant.html')

segment
⚠️ High Revenue, Low Satisfaction    340
💎 Hidden Gems                        340
⭐ Star Sellers                       279
🔴 At Risk                            278
Name: count, dtype: int64


In [21]:
segment_counts = seller_perf['segment'].value_counts().reset_index()
segment_counts.columns = ['Seller Group', 'Number of Sellers']

fig_simple = px.bar(segment_counts, 
    x='Seller Group', 
    y='Number of Sellers',
    title='📊 Overview of Our Seller Community',
    color='Seller Group',
    color_discrete_map={
        '⭐ Star Sellers': '#2ecc71',      
        '💎 Hidden Gems': '#3498db',      
        '⚠️ High Revenue, Low Satisfaction': '#e74c3c', 
        '🔴 At Risk': '#95a5a6'            
    },
    text_auto=True 
)

fig_simple.update_layout(template='plotly_white', showlegend=False)
fig_simple.show()


In [22]:
pay = payments.groupby('payment_type')['payment_value'].agg(['sum','count']).reset_index()
pay.columns = ['payment_type', 'total_value', 'transaction_count']
pay = pay[pay['payment_type'] != 'not_defined']

fig = make_subplots(rows=1, cols=2, specs=[[{'type':'pie'}, {'type':'pie'}]])
fig.add_trace(go.Pie(labels=pay['payment_type'], values=pay['transaction_count'],
    name='By Transaction', hole=0.4), row=1, col=1)
fig.add_trace(go.Pie(labels=pay['payment_type'], values=pay['total_value'],
    name='By Value', hole=0.4), row=1, col=2)

fig.update_layout(title='Payment Method: Transaction Count vs Total Value',
    template='plotly_white',
    annotations=[
        dict(text='By Count', x=0.18, y=0.5, font_size=12, showarrow=False),
        dict(text='By Value', x=0.82, y=0.5, font_size=12, showarrow=False)
    ])
fig.show()
fig.write_html('../outputs/10_payment_methods.html')

In [23]:
customer_orders = df.groupby('customer_unique_id' if 'customer_unique_id' in df.columns else 'customer_id')['order_id'].nunique().reset_index()
customer_orders.columns = ['customer_id', 'order_count']

repeat_rate = (customer_orders['order_count'] > 1).mean() * 100
print(f"Repeat Customer Rate: {repeat_rate:.1f}%")
print(f"One-time buyers: {(customer_orders['order_count'] == 1).sum():,}")
print(f"Repeat buyers: {(customer_orders['order_count'] > 1).sum():,}")

dist = customer_orders['order_count'].value_counts().sort_index().head(8).reset_index()
dist.columns = ['orders_placed', 'customer_count']

fig = px.bar(dist, x='orders_placed', y='customer_count',
    title=f'Customer Purchase Frequency (Repeat Rate: {repeat_rate:.1f}%)',
    labels={'orders_placed': 'Number of Orders', 'customer_count': 'Customers'},
    color_discrete_sequence=['#1a1a2e']
)
fig.update_layout(template='plotly_white')
fig.show()
fig.write_html('../outputs/11_repeat_customers.html')

Repeat Customer Rate: 3.0%
One-time buyers: 90,549
Repeat buyers: 2,801


In [24]:
state_revenue = df.groupby('customer_state').agg(
    revenue=('item_revenue', 'sum'),
    orders=('order_id', 'nunique'),
    avg_ticket=('item_revenue', 'mean')
).reset_index().sort_values('revenue', ascending=False)

fig = px.bar(state_revenue.head(15), x='customer_state', y='revenue',
    title='Revenue by Customer State (Top 15)',
    color='avg_ticket',
    color_continuous_scale='Blues',
    labels={'customer_state': 'State', 'revenue': 'Total Revenue (BRL)', 'avg_ticket': 'Avg Ticket'},
    text=state_revenue.head(15)['revenue'].apply(lambda x: f'BRL {x/1e6:.1f}M')
)
fig.update_traces(textposition='outside')
fig.update_layout(template='plotly_white')
fig.show()
fig.write_html('../outputs/12_revenue_by_state.html')

In [25]:
df['order_hour'] = df['order_purchase_timestamp'].dt.hour
df['order_dayofweek'] = df['order_purchase_timestamp'].dt.dayofweek

day_hour = df.groupby(['order_dayofweek', 'order_hour'])['order_id'].count().reset_index()
day_hour.columns = ['dayofweek', 'hour', 'orders']
day_hour_pivot = day_hour.pivot(index='dayofweek', columns='hour', values='orders').fillna(0)

day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
fig = px.imshow(day_hour_pivot,
    labels=dict(x='Hour of Day', y='Day of Week', color='Orders'),
    x=[str(h) for h in range(24)],
    y=day_names,
    title='🕐 When Do Customers Shop? (Order Heatmap)',
    color_continuous_scale='Blues'
)
fig.update_layout(template='plotly_white')
fig.show()
fig.write_html('../outputs/13_order_timing_heatmap.html')

In [26]:
freight = df.groupby('customer_state').agg(
    avg_freight=('freight_value', 'mean'),
    avg_price=('price', 'mean')
).reset_index()
freight['freight_pct'] = freight['avg_freight'] / freight['avg_price'] * 100
freight = freight.sort_values('freight_pct', ascending=False)

fig = px.bar(freight, x='customer_state', y='freight_pct',
    title='Freight Cost as % of Product Price by State',
    color='freight_pct',
    color_continuous_scale='OrRd',
    labels={'customer_state': 'State', 'freight_pct': 'Freight % of Price'}
)
fig.update_layout(template='plotly_white')
fig.show()
fig.write_html('../outputs/14_freight_by_state.html')


In [27]:
corr_cols = ['price', 'freight_value', 'payment_value', 'payment_installments',
             'delivery_days', 'review_score']
corr_df = df[corr_cols].dropna()
corr_matrix = corr_df.corr()

fig = px.imshow(corr_matrix, text_auto=True, aspect='auto',
    color_continuous_scale='RdBu_r',
    title='Correlation Matrix: Key Business Metrics',
    zmin=-1, zmax=1
)
fig.update_layout(template='plotly_white')
fig.show()
fig.write_html('../outputs/15_correlation_matrix.html')

In [28]:
from datetime import datetime

snapshot_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = df.groupby('customer_id').agg(
    recency=('order_purchase_timestamp', lambda x: (snapshot_date - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('item_revenue', 'sum')
).reset_index()

rfm['R_score'] = pd.qcut(rfm['recency'], q=4, labels=[4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=4, labels=[1,2,3,4])
rfm['M_score'] = pd.qcut(rfm['monetary'], q=4, labels=[1,2,3,4])
rfm['RFM_score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

print("RFM Score Distribution:")
print(rfm[['recency','frequency','monetary']].describe())
rfm.to_csv('../outputs/rfm_scores.csv', index=False)
print("RFM exported ✓")

RFM Score Distribution:
       recency  frequency  monetary
count 96470.00   96470.00  96470.00
mean    240.13       1.00    159.83
std     152.84       0.00    218.80
min       1.00       1.00      9.59
25%     116.00       1.00     61.85
50%     221.00       1.00    105.28
75%     350.00       1.00    176.26
max     714.00       1.00  13664.08
RFM exported ✓


In [29]:
# 1. Calculate growth
df['order_quarter'] = df['order_purchase_timestamp'].dt.to_period('Q')
q1_17_rev = df[df['order_quarter'] == '2017Q1']['item_revenue'].sum()
q3_18_rev = df[df['order_quarter'] == '2018Q3']['item_revenue'].sum()
growth_x = q3_18_rev / q1_17_rev if q1_17_rev > 0 else 0

# 2. Black Friday Spike
monthly_revs = df.groupby('order_month')['item_revenue'].sum()
oct_17 = monthly_revs.get('2017-10', 1)
nov_17 = monthly_revs.get('2017-11', 0)
bf_spike = nov_17 / oct_17

# 3. Top Category
top_cat = df.groupby('product_category_name_english')['item_revenue'].sum().idxmax()
top_cat = top_cat.title() if isinstance(top_cat, str) else "Unknown"

# 4. On-time Delivery
ot_rate = df['on_time'].mean() * 100

# 5. Late Deliveries Impact
late_avg_rating = df[df['delivery_days'] >= 21]['review_score'].mean()

# 6. Repeat Customer Rate
cust_col = 'customer_unique_id' if 'customer_unique_id' in df.columns else 'customer_id'
customer_counts = df.groupby(cust_col)['order_id'].nunique()
rep_rate = (customer_counts > 1).mean() * 100

# 7. SP & RJ Revenue
state_revs = df.groupby('customer_state')['item_revenue'].sum()
total_rev = state_revs.sum()
sp_rj_pct = ((state_revs.get('SP', 0) + state_revs.get('RJ', 0)) / total_rev) * 100 if total_rev > 0 else 0

# 8. Credit Card Dominance
cc_count_pct = (payments[payments['payment_type'] == 'credit_card']['payment_value'].count() / payments['payment_value'].count()) * 100

# 9. Peak Shopping Window
day_map = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
peak_day_num = df.groupby('order_dayofweek')['order_id'].count().idxmax()
peak_day = day_map.get(peak_day_num, 'Friday')
peak_hour = df.groupby('order_hour')['order_id'].count().idxmax()

# 10. High Revenue / Low Satisfaction Sellers
med_revenue = seller_perf['total_revenue'].median()
med_rating = seller_perf['avg_rating'].median()
high_rev_sellers = seller_perf[seller_perf['total_revenue'] >= med_revenue]
low_sat_high_rev = high_rev_sellers[high_rev_sellers['avg_rating'] < med_rating]
pct_low_sat = (len(low_sat_high_rev) / len(high_rev_sellers)) * 100 if len(high_rev_sellers) > 0 else 0

print("╔══════════════════════════════════════════════════════════════╗")
print("║           OLIST EDA — KEY BUSINESS FINDINGS                 ║")
print("╠══════════════════════════════════════════════════════════════╣")
print("║                                                              ║")
print(f"║  1. Revenue grew {growth_x:.1f}× from Q1 2017 to Q3 2018               ║")
print(f"║  2. Black Friday 2017 caused a ~{bf_spike:.1f}× monthly revenue spike     ║")
print(f"║  3. {top_cat:<10} is the top category by revenue       ║")
print(f"║  4. On-time delivery rate is {ot_rate:.1f}% nationally                 ║")
print(f"║  5. Late deliveries (21+ days) drop avg rating to {late_avg_rating:.1f}       ║")
print(f"║  6. Only {rep_rate:.1f}% of customers make repeat purchases              ║")
print(f"║  7. SP and RJ states account for {sp_rj_pct:.1f}% of all revenue         ║")
print(f"║  8. Credit card dominates: {cc_count_pct:.1f}% of transactions               ║")
print(f"║  9. {peak_day} at {peak_hour:02d}:00 is the peak shopping window             ║")
print(f"║  10. {pct_low_sat:.1f}% of high-revenue sellers have below-avg ratings      ║")
print("║                                                              ║")
print("╚══════════════════════════════════════════════════════════════╝")

╔══════════════════════════════════════════════════════════════╗
║           OLIST EDA — KEY BUSINESS FINDINGS                 ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. Revenue grew 2.5× from Q1 2017 to Q3 2018               ║
║  2. Black Friday 2017 caused a ~1.5× monthly revenue spike     ║
║  3. Health_Beauty is the top category by revenue       ║
║  4. On-time delivery rate is 92.1% nationally                 ║
║  5. Late deliveries (21+ days) drop avg rating to 3.1       ║
║  6. Only 3.0% of customers make repeat purchases              ║
║  7. SP and RJ states account for 50.7% of all revenue         ║
║  8. Credit card dominates: 73.9% of transactions               ║
║  9. Monday at 16:00 is the peak shopping window             ║
║  10. 54.9% of high-revenue sellers have below-avg ratings      ║
║                                                              ║
╚═════════════════════════

In [30]:
import plotly.io as pio

# Save your interactive chart as a high-resolution print image
fig.write_image('../outputs/seller_quadrant.png', width=1200, height=700, scale=2)
print("Image successfully exported to PNG! ✓")

Image successfully exported to PNG! ✓
